# 第 22 章 OpenCV 物件偵測

In [4]:
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('TkAgg') 
from opencv_tools import opencv_tools # 匯入封裝的功能
from ultralytics import YOLO

### 22-2-3 程式範例：透過 YOLO 進行物件偵測

* draw_boxes 函式：負責將偵測結果繪製在圖片上

In [5]:
def draw_boxes(image, results):
    result_image = image.copy()
    for box in results[0].boxes:
        # 取得座標、信心分數與類別名稱
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls_name = results[0].names[int(box.cls[0])]
        label = f"{cls_name}: {conf:.2f}"

        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 255, 0), 2,
                      lineType=cv2.LINE_AA)
        cv2.putText(result_image, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2,
                    lineType=cv2.LINE_AA)
    return result_image

* 主程式流程

In [6]:
# 載入模型、執行偵測並顯示結果
model = YOLO("yolov8n.pt")
image = cv2.imread("sample/yolo_sample.jpg")
results = model(image, conf=0.5, verbose=False)
result_image = draw_boxes(image, results)
opencv_tools.show_img_by_opencv(result_image) # 改用 OpenCV 視窗顯示

按下 Q 鍵，視窗即將關閉！


### 22-3-3 程式範例：從 YOLO 結果中篩選物件類別

In [7]:
def draw_boxes_filtered(image, results, target_classes):
    result_image = image.copy()
    for box in results[0].boxes:
        cls_name = results[0].names[int(box.cls[0])]

        # 不在篩選清單中的類別直接跳過
        if cls_name not in target_classes:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        label = f"{cls_name}: {conf:.2f}"

        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 255, 0), 2,
                      lineType=cv2.LINE_AA)
        cv2.putText(result_image, label, (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2,
                    lineType=cv2.LINE_AA)
    return result_image

In [8]:
target_classes = ["person", "car"]
image = cv2.imread("sample/yolo_sample.jpg")
results = model(image, conf=0.5, verbose=False)
result_image = draw_boxes_filtered(image, results, target_classes)
opencv_tools.show_img_by_opencv(result_image)

按下 Q 鍵，視窗即將關閉！


## 22-4 補充：YOLOv8 的不同大小模型

In [ ]:
# model = YOLO("yolov8n.pt")  # nano，最輕量
# model = YOLO("yolov8s.pt")  # small
# model = YOLO("yolov8m.pt")  # medium
# model = YOLO("yolov8l.pt")  # large
# model = YOLO("yolov8x.pt")  # extra large，最高精準度

## 22-E 小專案：即時攝影機 YOLO 偵測與生活應用

In [13]:
# --- 繪製函式（沿用 22-2-3 的 draw_boxes）---
def draw_boxes(image, results):
    result_image = image.copy()
    for box in results[0].boxes:
        # 取得座標、信心分數與類別名稱
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        cls_name = results[0].names[int(box.cls[0])]
        label = f"{cls_name}: {conf:.2f}"

        cv2.rectangle(result_image, (x1, y1), (x2, y2), (0, 255, 0), 2,
                      lineType=cv2.LINE_AA)
        cv2.putText(result_image, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2,
                    lineType=cv2.LINE_AA)
    return result_image

In [14]:
# --- 共用前處理函式 ---
def enhance_brightness(frame, alpha=1.2, beta=50):
    """調整影像的亮度與對比度，改善攝影機畫面偏暗的問題。"""
    return cv2.convertScaleAbs(frame, alpha=alpha, beta=beta)

In [23]:
# --- 設定區 ---
# 切換不同大小的模型（註解/取消註解切換）
model = YOLO("yolov8n.pt")    # 輕量版，速度優先
# model = YOLO("yolov8m.pt")  # 中型版，精準度優先

CONF_THRESHOLD = 0.5  # 信心分數門檻

# --- 主程式：即時攝影機偵測 ---
cap = cv2.VideoCapture(0)
# cap = cv2.VideoCapture("sample/video/me2.mp4")  # 影片測試
# cap = cv2.VideoCapture("sample/video/me3.mp4")  # 影片測試

# 用於計算即時 FPS
prev_time = cv2.getTickCount()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 前處理（視測試需求決定是否啟用）
    # frame = enhance_brightness(frame)

    # YOLO 物件偵測
    results = model(frame, conf=CONF_THRESHOLD, verbose=False)
    frame = draw_boxes(frame, results)

    # 計算 FPS 並顯示在畫面左上角
    curr_time = cv2.getTickCount()
    fps = cv2.getTickFrequency() / (curr_time - prev_time)
    prev_time = curr_time
    cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2,
                lineType=cv2.LINE_AA)

    # 顯示結果
    cv2.imshow("YOLO Real-time Detection", frame)

    # 按 q 鍵退出
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()